# YOLO11n — 704-image separate experiment

FLIC 104 + Canteen 175 + PUST 425. Grouped split: train 536 / val 97 / test 71. Provisional AI-reviewed labels only; not a field generalization test. Does **not** touch the frozen V1 specialists.

In Colab: **Runtime → Change runtime type → GPU**.

In [ ]:
import sys, subprocess, torch
print('CUDA available:', torch.cuda.is_available())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.175'], check=True)

## Upload the source ZIP

Download `YOLO_V2_704_GroupSplit_Trial.zip` from the YOLO Command Center chat, then upload it below. The 136 MB archive is **not** stored in GitHub.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
up = files.upload()
p = next((Path(x) for x in up if x.startswith('YOLO_V2_704_GroupSplit_Trial') and x.endswith('.zip')), None)
if p is None: raise ValueError('Upload YOLO_V2_704_GroupSplit_Trial.zip')
with ZipFile(p) as z: z.extractall('/content')
assert Path('/content/yolo_v2_trial_704/data.yaml').is_file()

## Verify and train

The packaged training script hashes all 704 images and labels and checks grouped split isolation. Model: **YOLO11n**, 35 epochs, internal held-out evaluation.

In [ ]:
import subprocess, sys, torch
script = '/content/yolo_v2_trial_704/train_yolo_v2_trial.py'
device = '0' if torch.cuda.is_available() else 'cpu'
batch = '8' if torch.cuda.is_available() else '4'
subprocess.run([sys.executable, script, '--verify-only'], check=True)
subprocess.run([sys.executable, script, '--device', device, '--batch', batch, '--epochs', '35'], check=True)

## Export metrics and best weights

In [ ]:
import json, zipfile
from pathlib import Path
from google.colab import files
base = Path('/content/YOLO_V2_trial_results')
summary = base/'test_summary.json'
print(json.dumps(json.loads(summary.read_text()),indent=2))
out = Path('/content/YOLO_V2_trial_results_only.zip')
with zipfile.ZipFile(out,'w',zipfile.ZIP_DEFLATED) as z:
    for rel in ['test_summary.json','experiment_config.json','train/weights/best.pt','train/results.csv']:
        f = base/rel
        if f.exists(): z.write(f,arcname=rel)
files.download(str(out))